# Enriquecimento e consolidação
Parto dos dois grupos gravados na limpeza e recupero somente campos ausentes.
O checkpoint contém respostas do OpenAlex já fornecidas ao projeto; não faço novas consultas.
A comparação antes/depois usa os mesmos artigos da execução atual.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd
from IPython.display import Markdown, display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))

from src import config as cfg
from src.dados import apply_metadata_checkpoint

pd.set_option("display.max_colwidth", 120)
from src.io_utils import atomic_json, atomic_parquet
cfg.ensure_directories()


## Abrir os grupos de entrada

Carrego os completos e incompletos gerados na limpeza. O grupo de incompletos é a população da análise de recuperação; os completos originais não precisam de preenchimento.

In [2]:
completos = pd.read_parquet(cfg.STAGED / "01_completas.parquet")
incompletos = pd.read_parquet(cfg.STAGED / "01_incompletas.parquet")
campos = cfg.COLUMNS[2:]
display(pd.Series({"Completos originais": len(completos), "Incompletos antes": len(incompletos)}, name="Artigos").to_frame())
display(incompletos[campos].isna().sum().rename("Ausências antes").to_frame())

,Artigos
Completos originais,529470
Incompletos antes,125313


,Ausências antes
title,495
abstract,121512
authors,17211
publish_time,1085


## O que significa uma consulta bem-sucedida?

Examino os status armazenados por DOI. Sucesso da consulta não significa que título, resumo, autores e data tenham sido todos retornados. O checkpoint também não informa a data individual de cada consulta; não atribuo a ele uma data inferida.

In [3]:
caminho_checkpoint = cfg.enrichment_path()
checkpoint = json.loads(caminho_checkpoint.read_text(encoding="utf-8"))
status = pd.Series([item.get("_status", "sem_status") for item in checkpoint.values()]).value_counts()
display(status.rename("Entradas no checkpoint").to_frame())
display(Markdown(f"O checkpoint contém **{len(checkpoint):,} entradas por DOI**. A recuperação abaixo é medida nos artigos e campos efetivamente preenchidos."))

,Entradas no checkpoint
sucesso,125129
nao_encontrado,115
erro,69


O checkpoint contém **125,313 entradas por DOI**. A recuperação abaixo é medida nos artigos e campos efetivamente preenchidos.

## Preencher somente ausências
Associo as respostas por DOI e comparo cada artigo pelo seu `cord_uid`.
A verificação abaixo protege os campos que já estavam preenchidos.

In [4]:
enriquecidos = apply_metadata_checkpoint(incompletos, checkpoint)
antes = incompletos.set_index("cord_uid")
depois = enriquecidos.set_index("cord_uid").reindex(antes.index)
preenchidos = antes[campos].isna() & depois[campos].notna()
alterados = antes[campos].notna() & ~antes[campos].eq(depois[campos]).fillna(False)
assert not alterados.any().any(), "Um campo originalmente preenchido foi alterado."
atomic_parquet(cfg.STAGED / "02_incompletas_enriquecidas.parquet", enriquecidos)
display(Markdown("O enriquecimento preencheu somente ausências e foi salvo para a consolidação."))

O enriquecimento preencheu somente ausências e foi salvo para a consolidação.

## Quantos campos foram recuperados?

A tabela reproduz a estrutura da Tabela 1 da seção 4.1, calculada diretamente nos mesmos artigos antes/depois. O percentual usa as ausências iniciais de cada campo. Somar campos recuperados não equivale a contar artigos recuperados.

In [5]:
recuperacao = pd.DataFrame({
    "Ausências antes": antes[campos].isna().sum(),
    "Informações recuperadas": preenchidos.sum(),
    "Ausências após": depois[campos].isna().sum(),
})
recuperacao["Recuperação (%)"] = (100 * recuperacao["Informações recuperadas"] /
    recuperacao["Ausências antes"].where(recuperacao["Ausências antes"].gt(0))).fillna(0).round(2)
recuperacao.index = recuperacao.index.map({"title": "Título", "abstract": "Resumo", "authors": "Autores", "publish_time": "Data de publicação"})
display(recuperacao)

,Ausências antes,Informações recuperadas,Ausências após,Recuperação (%)
Título,495,493,2,99.60
Resumo,121512,51146,70366,42.09
Autores,17211,2618,14593,15.21
Data de publicação,1085,1085,0,100.00


## Ver exemplos reais de preenchimento e de limitações

Seleciono deterministicamente até cinco casos de campos recuperados e cinco artigos ainda incompletos. A exibição abrevia textos longos; os valores usados no processamento continuam inteiros. Estes exemplos ilustram a comparação e não são uma nova amostra de modelagem.

In [6]:
exemplos = []
for campo in campos:
    ids = preenchidos.index[preenchidos[campo]].sort_values()[:2]
    for uid in ids:
        exemplos.append({"cord_uid": uid, "Campo": campo, "Antes": antes.at[uid, campo], "Depois": depois.at[uid, campo]})
display(pd.DataFrame(exemplos).sort_values(["cord_uid", "Campo"]).head(5))
ainda_incompletos = depois.loc[depois[campos].isna().any(axis=1)]
display(ainda_incompletos.sort_index().head(5))

,cord_uid,Campo,Antes,Depois
2,00012cqv,abstract,NaN,"As our population ages, there is an increasing need to develop effective, scalable interventions to support health a..."
3,000xcuss,abstract,NaN,"Dear Editor, According to recent studies, approximately 33% of patients hospitalized with coronavirus disease 2019 (..."
0,003kzajt,title,NaN,Revisión rápida de la transmisión del SARS-CoV-2 por contacto con objetos y superficies
4,005stnnr,authors,NaN,American Diabetes Association
5,00fxzyhq,authors,NaN,American Geriatrics Society


,doi,title,abstract,authors,publish_time
cord_uid,,,,,
0007wzmf,10.1007/s00404-022-06422-6,Trends in the risk of second primary malignancies among survivors of ovarian clear cell carcinoma,NaN,"Wang, Yiru; Guo, Ying; Li, Haiyan",2022
0011irx9,10.1016/s2213-2600(21)00123-5,Young person with long COVID after mild disease,NaN,"Kirby, Tony",2021-05-31
001wbz6e,10.1016/0958-1669(95)80119-7,Pharmaceutical biotechnology,NaN,NaN,2002-02-25
0021yo3o,10.1016/j.jinf.2010.07.005,Procalcitonin in children with suspected novel influenza A (H1N1) infection,NaN,"Limper, Maarten; Smit, Patrick M.; Bongers, Karlien M.; van Zanten, Anton P.; Smits, Paul H.M.; Brandjes, Dees P.M.;...",2010-07-16
002le26o,10.1016/j.eimce.2021.08.015,Effectiveness of mRNA Covid-19 vaccine in healthcare workers,NaN,"Sookaromdee, Pathum; Wiwanitkit, Viroj",2022-05-13


## Diferenciar campos, artigos recuperados e novos completos

Um artigo pode receber vários campos e continuar incompleto. Por isso calculo separadamente os campos preenchidos, os artigos com alguma recuperação e os artigos que passaram a satisfazer todos os requisitos.

In [7]:
artigos_com_recuperacao = int(preenchidos.any(axis=1).sum())
novos_completos = int(depois[cfg.COLUMNS[1:]].notna().all(axis=1).sum())
resumo = pd.Series({
    "Campos preenchidos": int(preenchidos.sum().sum()),
    "Artigos com alguma recuperação": artigos_com_recuperacao,
    "Artigos que ficaram completos": novos_completos,
    "Artigos ainda incompletos": len(ainda_incompletos),
}, name="Quantidade")
display(resumo.to_frame())
display(Markdown(f"**{100 * novos_completos / len(incompletos):.2f}%** dos artigos inicialmente incompletos ficaram completos. Recuperar uma informação não garante a inclusão do artigo no consolidado."))

,Quantidade
Campos preenchidos,55342
Artigos com alguma recuperação,54445
Artigos que ficaram completos,51545
Artigos ainda incompletos,73768


**41.13%** dos artigos inicialmente incompletos ficaram completos. Recuperar uma informação não garante a inclusão do artigo no consolidado.

## Consolidar os registros completos
Seleciono os artigos enriquecidos que ficaram completos, junto aos completos originais.
Mantenho uma ocorrência por identificador e gravo o consolidado atual.

In [8]:
mascara = enriquecidos[cfg.COLUMNS].notna().all(axis=1)
novos = enriquecidos.loc[mascara]
descartados = enriquecidos.loc[~mascara]
consolidado = pd.concat([completos, novos], ignore_index=True).drop_duplicates("cord_uid", keep="first")
consolidado = consolidado.reset_index(drop=True)
atomic_parquet(cfg.STAGED / "03_consolidado.parquet", consolidado)
consolidacao = {"Completos originais": len(completos), "Novos completos": len(novos),
                "Consolidado": len(consolidado), "Descartados por incompletude": len(descartados)}
display(pd.Series(consolidacao, name="Artigos").to_frame())
display(Markdown(f"O enriquecimento ampliou os completos em **{100 * len(novos) / len(completos):.2f}%**."))

,Artigos
Completos originais,529470
Novos completos,51545
Consolidado,581015
Descartados por incompletude,73768


O enriquecimento ampliou os completos em **9.74%**.

## Fechamento
O consolidado salvo é a entrada do recorte temporal. A disponibilidade de metadados
limita o universo que poderá ser analisado.